In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/BankChurners.csv')

colunas_naive_bayes = [col for col in df.columns if col.startswith('Naive_Bayes')]
df = df.drop(columns=['CLIENTNUM'] + colunas_naive_bayes, errors='ignore')

df.shape

(10127, 20)

In [3]:
df['Churn'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)

df = df.drop(columns=['Attrition_Flag'])

df['Churn'].value_counts()

Churn
0    8500
1    1627
Name: count, dtype: int64

In [4]:
colunas_categoricas = df.select_dtypes(include=['object', 'string']).columns.tolist()
print(colunas_categoricas)


['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


In [5]:
df = pd.get_dummies(df, columns=colunas_categoricas, drop_first=True)

df.shape

(10127, 33)

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} clientes')
print(f'Teste: {X_test.shape[0]} clientes')

Treino: 8101 clientes
Teste: 2026 clientes


In [7]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

print('Modelo treinado com sucesso!')

Modelo treinado com sucesso!


c:\Users\breno\OneDrive\Área de Trabalho\Projeto linkedin\-churn-cartao-credito-b2c\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = modelo.predict(X_test)

print(classification_report(y_test,y_pred, target_names=['Não Churn', 'Churn']))

              precision    recall  f1-score   support

   Não Churn       0.91      0.97      0.94      1701
       Churn       0.77      0.52      0.62       325

    accuracy                           0.90      2026
   macro avg       0.84      0.74      0.78      2026
weighted avg       0.89      0.90      0.89      2026



In [9]:
modelo_balanceado = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
modelo_balanceado.fit(X_train, y_train)

y_pred_balanceado = modelo_balanceado.predict(X_test)
print(classification_report(y_test, y_pred_balanceado, target_names=['Não Churn', 'Churn']))

              precision    recall  f1-score   support

   Não Churn       0.95      0.84      0.89      1701
       Churn       0.49      0.78      0.60       325

    accuracy                           0.83      2026
   macro avg       0.72      0.81      0.75      2026
weighted avg       0.88      0.83      0.85      2026



c:\Users\breno\OneDrive\Área de Trabalho\Projeto linkedin\-churn-cartao-credito-b2c\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo_balanceado.fit(X_train_scaled, y_train)
y_pred_balanceado = modelo_balanceado.predict(X_test_scaled)
print(classification_report(y_test, y_pred_balanceado, target_names=['Não Churn', 'Churn']))

              precision    recall  f1-score   support

   Não Churn       0.96      0.86      0.91      1701
       Churn       0.53      0.82      0.64       325

    accuracy                           0.85      2026
   macro avg       0.75      0.84      0.78      2026
weighted avg       0.89      0.85      0.87      2026



In [11]:
coeficientes = pd.DataFrame({
    'variavel': X.columns,
    'coeficiente': modelo_balanceado.coef_[0]
}).sort_values('coeficiente', key=abs, ascending=False)

coeficientes.head(10)

,variavel,coeficiente
11,Total_Trans_Ct,-3.024478
10,Total_Trans_Amt,1.790733
12,Total_Ct_Chng_Q4_Q1,-0.606237
7,Total_Revolving_Bal,-0.604718
3,Total_Relationship_Count,-0.601669
5,Contacts_Count_12_mon,0.579679
4,Months_Inactive_12_mon,0.557786
14,Gender_M,-0.402141
27,Income_Category_Less than $40K,-0.226488
24,Income_Category_$40K - $60K,-0.214332


In [12]:
# Gera previsões de probabilidade de churn para toda a base de teste
probabilidades = modelo_balanceado.predict_proba(X_test_scaled)[:, 1]

# Monta a tabela final: identificador do cliente + probabilidade prevista
df_previsoes = pd.DataFrame({
    'indice_cliente': X_test.index,
    'probabilidade_churn': probabilidades,
    'churn_real': y_test.values
})

df_previsoes = df_previsoes.sort_values('probabilidade_churn', ascending=False)

df_previsoes.to_csv('../data/processed/previsoes_churn.csv', index=False)

df_previsoes.head(10)

,indice_cliente,probabilidade_churn,churn_real
24,8574,0.998600,1
1152,6244,0.998012,1
1278,5925,0.997751,1
1095,1830,0.997727,1
1709,8120,0.996960,1
1274,4406,0.996772,1
600,2414,0.996244,1
1424,1693,0.996131,1
469,8959,0.995875,1
797,4926,0.995229,1
